In [ ]:
%pip install --upgrade pip

In [ ]:
%pip install -r ../requirements.txt
%pip install scikit-learn

In [ ]:
%pip install openpyxl
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

file_path = 'C:\\Users\\sweth\\ML Year 4\\bangladesh_dataset\\Survey-Survey-cleaned.csv.xlsx'
try:
    df = pd.read_excel(file_path)
except Exception:
    df = pd.read_csv(file_path)

print("Columns in dataset:")
for idx, col in enumerate(df.columns):
    print(f"{idx}: {col}")

In [ ]:


tipi_rename_map = {
    '1. _____ Extraverted, enthusiastic.': 'TIPI_1',
    '2. _____ Critical, quarrelsome.': 'TIPI_2',
    '3. _____ Dependable, self-disciplined.': 'TIPI_3',
    '4. _____ Anxious, easily upset.': 'TIPI_4',
    '5. _____ Open to new experiences, complex.': 'TIPI_5',
    '6. _____ Reserved, quiet.': 'TIPI_6',
    '7. _____ Sympathetic, warm.': 'TIPI_7',
    '8. _____ Disorganized, careless.': 'TIPI_8',
    '9. _____ Calm, emotionally stable': 'TIPI_9',
    '10. _____ Conventional, uncreative.': 'TIPI_10'
}

dass_rename_map = {
    '1. I found it hard to wind down': 'DASS_1',
    '2. I tended to over-react to situations': 'DASS_2',
    '3 .I felt that I was using a lot of nervous energy': 'DASS_3',
    '4.  I found myself getting agitated': 'DASS_4',
    '5.  I found it difficult to relax': 'DASS_5',
    '6.  I was intolerant of anything that kept me from getting on with what I was doing': 'DASS_6',
    '7.  I felt that I was rather touchy': 'DASS_7'
}

df = df.rename(columns=tipi_rename_map)
df = df.rename(columns=dass_rename_map)

tipi_cols = [f'TIPI_{i}' for i in range(1, 11)]
stress_cols = [f'DASS_{i}' for i in range(1, 8)]

def clean_text(text):
    if pd.isna(text): return 'Unknown'
    return str(text).strip().lower()

program_standard_map = {
    'cse': 'Computer Science & Engineering', 'computer science': 'Computer Science', 'cs': 'Computer Science',
    'eee': 'Electrical & Electronic Engineering', 'bba': 'Business Administration', 'physics': 'Physics',
    'law': 'Law', 'llb': 'Law', 'english': 'English', 'civil': 'Civil Engineering',
    'mechanical': 'Mechanical Engineering', 'textile': 'Textile Engineering', 'chemistry': 'Chemistry',
    'botany': 'Botany', 'zoology': 'Zoology', 'pharmacy': 'Pharmacy', 'history': 'History',
    'sociology': 'Sociology', 'economics': 'Economics'
}

institution_standard_map = {
    'edu': 'East Delta University', 'east delta': 'East Delta University',
    'chittagong university': 'University of Chittagong', 'cu': 'University of Chittagong',
    'iiuc': 'International Islamic University Chittagong', 'cvasu': 'CVASU', 'bgc': 'BGC Trust University',
    'ciu': 'Chittagong Independent University', 'cuet': 'CUET', 'dhaka university': 'University of Dhaka',
    'du': 'University of Dhaka', 'aiub': 'AIUB', 'buet': 'BUET'
}

def map_program(val):
    clean = clean_text(val)
    for keyword, standard in program_standard_map.items():
        if keyword in clean: return standard
    return 'Other/General'

def map_institution(val):
    clean = clean_text(val)
    for keyword, standard in institution_standard_map.items():
        if keyword in clean: return standard
    return 'Other/General'

df['Program_Cleaned'] = df['The name of your program of study'].apply(map_program)
df['Institution_Cleaned'] = df['The name of your institution'].apply(map_institution)

class_level_mapping = {
    "Freshman (Undergrad)": 0, "Sophomore (Undergrad)": 1,
    "Junior (Undergrad)": 2, "Senior (Undergrad)": 3, "Graduate student": 4
}
df['class_level_num'] = df['Your current class level is'].map(class_level_mapping)
df['class_level_num'] = df['class_level_num'].fillna(df['class_level_num'].median())
print("Data loaded and initial features cleaned successfully.")

In [ ]:
#LIFESTYLE & DEMOGRAPHIC FEATURES

def norm(val):
    """Lower-case and remove spaces so 'Yes ', 'yes' and 'YES' all match."""
    if pd.isna(val):
        return np.nan
    return str(val).strip().lower().replace(' ', '')

binary_map = {'no': 0, 'yes': 1}

lifestyle_specs = {
    'Living with family?':                               ('family_num',           binary_map),
    'Are you happy with your academic  Condition?':      ('academic_happy_num',   binary_map),
    'Are you addicted to any drugs?':                    ('drugs_num',            binary_map),
    'Are you in a relationship?':                        ('relationship_num',     binary_map),
    'Did you have a recent breakup?':                    ('breakup_num',          binary_map),
    'Do you have financial problem in your family?':     ('finance_num',          binary_map),
    'Have you ever been bullied':                        ('bullied_num',          binary_map),
    'Have you ever been sexually harassed or abused?':   ('harassed_num',         binary_map),
    'Your current age':                                  ('age_num',              {'<20': 0, '20-25': 1, '21-25': 1, '>25': 2}),
    'Your CGPA':                                         ('cgpa_num',             {'<3.00': 0, '3.01-3.50': 1, '>3.50': 2}),
    'How often do you conflict with your friend?':       ('conflict_num',         {'neveroccurs': 0, 'never': 0, 'sometimes': 1, 'mostofthetime': 2}),
    'Violence in family?':                               ('violence_num',         {'never': 0, 'rarely': 1, 'often': 2, 'mostofthetime': 3}),
    'How many hours do you spend on social media?':      ('social_media_num',     {'<1hours': 0, '<1hour': 0, '1-3hours': 1, '1-3hour': 1, '>3hours': 2, '>3hour': 2}),
}

for raw_col, (num_col, mapping) in lifestyle_specs.items():
    normed = df[raw_col].apply(norm)
    df[num_col] = normed.map(mapping)

    unmapped = df.loc[normed.notna() & df[num_col].isna(), raw_col].unique()
    if len(unmapped) > 0:
        print(f"WARNING - '{raw_col}' has values not in the mapping: {list(unmapped)}")

    df[num_col] = df[num_col].fillna(df[num_col].median())

lifestyle_raw_cols = list(lifestyle_specs.keys())
lifestyle_num_cols = [num_col for num_col, _ in lifestyle_specs.values()]

print("\nEncoded lifestyle features (check the value counts look sensible):")
for raw_col, (num_col, _) in lifestyle_specs.items():
    print(f"\n{num_col}  <-  {raw_col}")
    print(df[num_col].value_counts().sort_index().to_string())


In [ ]:
import matplotlib.pyplot as plt

# Select numeric variables for descriptive statistics
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

# Summary statistics
descriptive_stats = df[numeric_cols].describe().T
descriptive_stats["missing"] = df[numeric_cols].isna().sum()
display(descriptive_stats.round(2))

# Histograms
df[numeric_cols].hist(
    figsize=(18, 14),
    bins=15,
    edgecolor="black",
    layout=(-1, 4)
)
plt.suptitle("Distributions of Numeric Variables", fontsize=16)
plt.tight_layout()
plt.show()

# Boxplots for detecting outliers
plt.figure(figsize=(18, 8))
df[numeric_cols].boxplot(rot=90)
plt.title("Boxplots of Numeric Variables")
plt.ylabel("Value")
plt.tight_layout()
plt.show()

# Missing-value visualization
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if not missing.empty:
    missing.plot(kind="bar", figsize=(14, 5), color="tomato")
    plt.title("Missing Values by Variable")
    plt.ylabel("Number of Missing Values")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")